# bench_report

> Summaries and the auto-written report for the Phase 8 benchmark: per-size estimates, scaling slopes, side-sweep tables, the team-optimal exploration level, study B curves and gap closed.

Spec: `specs/multiagent/phase8/` (design-low-level section 6).

In [ ]:
#| default_exp bench_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import warnings

import numpy as np

from explRL.bench_stats import bootCI, scalingFit

CH = ('none', 'blackboard', 'messages', 'hierarchical')
DEFAULTS = dict(scheme='individual', lam=0.01, radius='inf', K='none', shape='ring', cUCB=1.0)


def _mean(v):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        return float(np.nanmean(np.asarray(v, float))) if len(v) else float('nan')


def _sel(rows, **kw):
    return [r for r in rows if all(r.get(k) == v for k, v in kw.items())]


def byN(rows, metric, **kw):
    "Per team size: (N, mean, CI low, CI high) of `metric` over the selected rows."
    sel = _sel(rows, **kw)
    return np.array([(N, _mean(v), *bootCI(v)) for N in sorted({int(r['N']) for r in sel})
                     for v in [[float(r[metric]) for r in sel if int(r['N']) == N]]]).reshape(-1, 4)


def slopes(rows):
    "Scaling fit of welfare per agent-second on log2 N, per channel and budget (main grid)."
    out = {}
    for b in ('compute', 'time'):
        for ch in CH:
            sel = _sel(rows, study='main', channel=ch, budget=b)
            if len({r['N'] for r in sel}) > 1:
                out[f'{ch}/{b}'] = scalingFit([r['N'] for r in sel], [r['perAgentSecond'] for r in sel], [r['seed'] for r in sel])
    return out


def _order(levels):
    def key(x):
        try:
            return (0, float(x), '')
        except ValueError:
            return (1, 0.0, x)
    return sorted(levels, key=key)


def sweepTable(rows):
    "Side sweeps: {factor: {level: {(channel, N): mean welfare per agent-second}}}, baseline as each factor's default."
    sw, out = _sel(rows, study='sweep'), {}
    for factor in sorted({r['factor'] for r in sw} - {'baseline', 'circuit'}):
        chans = {r['channel'] for r in _sel(sw, factor=factor)}
        sets = {str(DEFAULTS.get(factor, '-')): _sel(sw, factor='baseline')}
        sets.update({lv: _sel(sw, factor=factor, level=lv) for lv in {r['level'] for r in _sel(sw, factor=factor)}})
        out[factor] = {lv: {(ch, N): _mean([r['perAgentSecond'] for r in sets[lv] if r['channel'] == ch and int(r['N']) == N])
                            for ch in sorted(chans) for N in sorted({int(r['N']) for r in sets[lv]})}
                       for lv in _order(sets)}
    return out


def teamOptimal(table):
    "Team-optimal cUCB per (channel, N) from the cUCB sweep."
    lv = table.get('cUCB', {})
    keys = set().union(*[set(v) for v in lv.values()]) if lv else set()
    return {f'{ch}/{N}': max(lv, key=lambda l: lv[l].get((ch, N), -np.inf)) for ch, N in sorted(keys)}


def studyB(rows):
    "Per arm and N: mean welfare per episode, and the gap to the best fixed level closed over the last 4 episodes."
    sb, out = _sel(rows, study='studyB'), {}
    for N in sorted({int(r['N']) for r in sb}):
        rs = [r for r in sb if int(r['N']) == N]
        E = int(max(r['episode'] for r in rs)) + 1
        late = {lv: {s: _mean([r['perAgentSecond'] for r in rs if r['level'] == lv and r['seed'] == s and r['episode'] >= E - 4])
                     for s in sorted({r['seed'] for r in rs})} for lv in sorted({r['level'] for r in rs})}
        W = {lv: _mean(list(v.values())) for lv, v in late.items()}
        fixed = [lv for lv in W if lv.startswith('fixed')]
        best, base = (max(fixed, key=W.get), 'fixed-1|beta=0.0') if fixed else (None, None)
        den = [late[best][s] - late[base][s] for s in late.get(base, {})] if best and base in late else []
        ok = len(den) > 1 and (bootCI(den)[0] > 0)
        for lv in W:
            curve = [_mean([r['perAgentSecond'] for r in rs if r['level'] == lv and r['episode'] == e]) for e in range(E)]
            gap = (W[lv] - W[base]) / (W[best] - W[base]) if ok else float('nan')
            stable = _mean([r['levelStable'] for r in rs if r['level'] == lv and r['episode'] >= E - 4])
            out[f'{lv}/N={N}'] = dict(curve=curve, late=W[lv], gapClosed=gap, bestFixed=best, levelStable=stable)
    return out


def _f(x, d=3):
    return 'nan' if x != x else f'{x:.{d}f}'


def report(meta, conf=None, slope=None, table=None, opt=None, sb=None):
    "The benchmark's report.md text; everything but the confirmatory table is labelled exploratory."
    L = [f"# Phase 8 benchmark ({meta['mode']} run)\n",
         f"{meta['cells']} cells, {meta['workers']} worker processes, {meta['wallSeconds']:.0f} s. "
         f"Pre-registration hash `{meta['preregHash']}`, git commit `{meta['gitCommit']}`. "
         f"Deviations from the pre-registration: {meta.get('deviations', 'none')}.\n"]
    if conf:
        L.append("## Confirmatory results\n")
        if meta['mode'] == 'quick':
            L.append("*Quick run on pilot seeds with scaled budgets: a pipeline check, not a result.*\n")
        L.append("| test | contrast | mean [95% CI] | dz | p | Holm p | verdict |\n|---|---|---|---|---|---|---|")
        for k, c in conf.items():
            t = meta['prereg']['tests'][k]
            L.append(f"| {k} | {t['metric']}: {t['a']} - {t['b']}, sizes {t['sizes'][0]}-{t['sizes'][-1]} | "
                     f"{_f(c['mean'])} [{_f(c['ci'][0])}, {_f(c['ci'][1])}] | {_f(c['dz'], 2)} | {_f(c['p'], 4)} | "
                     f"{_f(c['pHolm'], 4)} | **{c['verdict']}** |")
        L.append("")
    if slope:
        L.append("## Scaling (exploratory)\n\nSlope of welfare per agent-second per doubling of team size [95% CI].\n")
        L += [f"- {k}: {_f(v['slope'], 4)} [{_f(v['ci'][0], 4)}, {_f(v['ci'][1], 4)}]" for k, v in slope.items()]
        L.append("")
    if table:
        L.append("## Side sweeps (exploratory)\n\nWelfare per agent-second; columns are channel / team size; "
                 "the factor's default level is the shared baseline.\n")
        for factor, lv in table.items():
            cols = sorted(set().union(*[set(v) for v in lv.values()]))
            L.append(f"**{factor}**\n\n| level | " + ' | '.join(f'{c}/{n}' for c, n in cols) + " |\n|---|" + '---|' * len(cols))
            L += [f"| {l} | " + ' | '.join(_f(v.get(c, float('nan'))) for c in cols) + " |" for l, v in lv.items()]
            L.append("")
    if opt:
        L.append("## Study A: team-optimal exploration level (exploratory)\n")
        L += [f"- {k}: cUCB {v}" for k, v in opt.items()] + [""]
    if sb:
        L.append("## Study B: persistence and credit (exploratory)\n\n"
                 "| arm / N | last-4 welfare | gap closed | level stable |\n|---|---|---|---|")
        L += [f"| {k} | {_f(v['late'])} | {_f(v['gapClosed'], 2)} | {_f(v['levelStable'], 2)} |" for k, v in sb.items()]
        L.append("\nGap closed: (arm - fixed-1) / (best fixed - fixed-1) over the last 4 episodes; nan when the best "
                 "fixed level is not clearly better than fixed-1.\n")
    L.append("Re-run: `nbdev-test --n-workers 0 nbs/demo_bench.ipynb` (quick), `--flags slow` (main grid), "
             "`--flags sweeps` (sweeps and studies). Spec: `specs/multiagent/phase8/`.\n")
    return '\n'.join(L)

## Tests

In [ ]:
import numpy as np
from fastcore.test import test_eq, test_close
from explRL.bench_grid import PREREG, preregHash, mainGrid, sideSweeps, runGrid, confirm
from explRL.bench_report import byN, slopes, sweepTable, teamOptimal, studyB, report, _order

test_eq(_order(['0.1', '0.01', 'none', '0.003', '1']), ['0.003', '0.01', '0.1', '1', 'none'])
rows = runGrid(mainGrid('quick'), workers=2) + runGrid(sideSweeps('quick'), workers=2)
a = byN(rows, 'perAgentSecond', study='main', channel='blackboard', budget='compute')
test_eq(a[:, 0].tolist(), [1.0, 4.0, 16.0]); assert (a[:, 2] <= a[:, 1]).all() and (a[:, 1] <= a[:, 3]).all()
sl = slopes(rows); test_eq(len(sl), 8)
tab = sweepTable(rows)
test_eq(set(tab), {'K', 'cUCB', 'lam', 'radius', 'scheme', 'shape'})
test_eq(list(tab['cUCB']), ['0.25', '1.0'])                        # quick: first level and the default (baseline)
opt = teamOptimal(tab); assert all(v in ('0.25', '1.0') for v in opt.values())
sb = studyB(rows); test_eq(list(sb), ['strategy|beta=1.0/N=4']); test_eq(len(sb['strategy|beta=1.0/N=4']['curve']), 2)
conf = confirm(rows, seeds=(1000, 1001))
meta = dict(mode='quick', cells=10, workers=2, wallSeconds=1.0, preregHash=preregHash(), gitCommit='abc', prereg=PREREG)
txt = report(meta, conf, sl, tab, opt, sb)
for s in ('## Confirmatory results', 'pipeline check', '| P |', '## Scaling', '## Side sweeps', '## Study A', '## Study B'):
    assert s in txt, s

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()